In [ ]:
# hide
# no-output
from IPython.utils.io import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly

In [ ]:
# hide
# autorun
def figure():
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.2)

    # left: the signal wound around the complex plane
    fig.add_scatter(name="Wound signal", x=[0], y=[0], mode="lines+markers",
                    marker=dict(size=3), row=1, col=1)
    # left: the center of mass (proportional to the DFT bin)
    fig.add_scatter(name="Center of mass", x=[0], y=[0], mode="markers",
                    marker=dict(size=14, color="#C41230"), row=1, col=1)
    # right: the real input sinusoid
    fig.add_scatter(name="Input", x=[0], y=[0], mode="lines", row=1, col=2)

    fig.update_xaxes(range=[-1.2, 1.2], scaleanchor="y", scaleratio=1, row=1, col=1,
                     title_text="Real", fixedrange=True)
    fig.update_yaxes(range=[-1.2, 1.2], scaleanchor="x", scaleratio=1, row=1, col=1,
                     title_text="Imaginary", fixedrange=True)
    fig.update_xaxes(range=[0, 4], row=1, col=2, title_text="Time (s)", fixedrange=True)
    fig.update_yaxes(range=[-1.2, 1.2], row=1, col=2, title_text="Amplitude", fixedrange=True)
    fig.update_layout(showlegend=False)
    return fig


def controls(fig):
    full = {"description_width": "initial"}
    freq_real = widgets.FloatSlider(description="Input frequency (Hz)", min=0.0, max=10.0,
                                    value=4.0, step=0.01, style=full)
    freq_probe = widgets.FloatSlider(description="Probe frequency (Hz)", min=0.0, max=10.0,
                                     value=4.0, step=0.05, style=full)
    sample_rate = widgets.IntSlider(description="Sample rate (Hz)", min=20, max=200,
                                    value=100, step=10, style=full)
    num_samples = widgets.IntSlider(description="Number of samples N", min=50, max=800,
                                    value=400, step=50, style=full)

    # redraw whenever any slider changes
    def update(freq_real, freq_probe, sample_rate, num_samples):
        n = np.arange(num_samples)
        t = n / sample_rate
        x = np.cos(2 * np.pi * freq_real * t)                 # real input sinusoid
        wound = x * np.exp(-1j * 2 * np.pi * freq_probe * t)  # multiply by the probe phasor
        com = wound.mean()                                    # center of mass (~ DFT bin)
        with fig.batch_update():
            fig.data[0].x = wound.real
            fig.data[0].y = wound.imag
            fig.data[1].x = [com.real]
            fig.data[1].y = [com.imag]
            fig.data[2].x = t
            fig.data[2].y = x
            fig.layout.xaxis2.range = [0, float(t[-1]) if len(t) else 1]

    widgets.interactive_output(update, {"freq_real": freq_real, "freq_probe": freq_probe,
                                        "sample_rate": sample_rate, "num_samples": num_samples})
    return widgets.VBox([freq_real, freq_probe, sample_rate, num_samples])


icm_plotly.show(figure, controls)